In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json, yaml
from functional_recovery import recover_local
runtime = json.loads(Path(RUNTIME_PATH).read_text())
config = yaml.safe_load(Path(runtime["config"]).read_text())
print(json.dumps({"config_path": runtime["config"], "config_sha256": runtime["config_sha256"], "seeds": config["shared"]["model_seeds"], "git_sha": runtime["git_sha"], "code_sha256": runtime["code_sha256"], "run_id": runtime["run_id"], "party": runtime["party"], "run_at": datetime.now(timezone.utc).isoformat()}, ensure_ascii=False))

In [ ]:
recover_local(runtime)

In [ ]:
if runtime["party"] == "alice":
    import pandas as pd
    report = json.loads((Path("/srv/vfl") / "alice" / runtime["run_id"] / "results/private_evaluation.json").read_text())
    rows = []
    for item in report["metrics"]:
        for name, value in item["metrics"].items():
            rows.append({"seed": item["seed"], "route": item["route"], "metric": name, **value})
    display(pd.DataFrame(rows))
else:
    print("Bob: no plaintext labels, gradients, scores or joint weights released")

本 Notebook 仅在新目录恢复已完成模型的收尾验收和评价。先验证原输入哈希、处理数组相同、训练代码指纹和所有候选完整；不重新训练、不调参、不修改失败来源目录。实际联邦计算和认证证据来自 origin_run_id，预测仍仅 Alice 获授权。